# Inspect saved data and models
Run `01_run.ipynb` (or `python run.py run --profile smoke`) first. This notebook does not train. `restore` opens a run with its saved model, data and analysis settings, also on CPU for a GPU-trained model. Move the entire output folder together: artifact references are relative.

In [ ]:
from pathlib import Path
import os
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "settings_training.py").is_file() and (p / "nnpd").is_dir())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from nnpd import load_settings, restore
from nnpd.results import runs

PROFILE = "smoke"  # or "default" / "paper"; use the same profile in all three notebooks
OUTPUT = load_settings("settings_training.py", PROFILE)["output"]

In [ ]:
records = runs(OUTPUT)
assert records, f"Run 01_run.ipynb or python run.py run --profile {PROFILE} first."
context = restore(records[0]["path"], device="cpu")
context.model

In [ ]:
observations = context.require("observations")
inference = context.require("inference")
{"truth_shape": observations.array("truth").shape,
 "observation_shape": observations.array("observations").shape,
 "mode_shape": inference.array("ratio_mode").shape,
 "metadata": inference.metadata}

## Call a metric or plot directly
Every hook is an ordinary Python function. Arrays are numeric `.npy` files, normally opened as read-only memory maps. Full candidate scores are retained only when requested in `settings_analysis.py`; the default retains diagnostic cases and all per-case summaries.

In [ ]:
hook = context.analysis.metrics()["coverage"]
hook.compute(context, context.dependencies(hook.needs))

In [ ]:
from IPython.display import display
from matplotlib import pyplot as plt
hook = context.analysis.plots()["inference"]
figures = hook.draw(context, context.dependencies(hook.needs))
display(next(iter(figures.values())))
for figure in figures.values():
    plt.close(figure)

In [ ]:
from nnpd.results import export_csv
export_csv(OUTPUT, Path(OUTPUT) / "comparison.csv",
           {"prior": "member.prior", "dimension": "settings.problem.dimension",
            "auc": "metrics.classification.auc", "mean_absolute_bias": "metrics.bias.mean_absolute"})